#**Practical No. 9**

**Aim:**
Build an end-to-end pipeline: CSV → Python/Pandas → Data Cleaning → SQL Database → Report.
Build an ETL pipeline for an e-commerce dataset containing customers, orders, products, and payments.
Design a data pipeline that handles both historical and newly arriving data.


In [1]:
import pandas as pd
import sqlite3

# 1. Create e-commerce data
customers = pd.DataFrame({
    "customer_id": [1, 2, 3, 4, 5],
    "name": ["Amit", "Priya", "Rahul", "Neha", "Rohan"],
    "city": ["Mumbai", "Pune", "Delhi", "Nashik", "Mumbai"]
})

products = pd.DataFrame({
    "product_id": [101, 102, 103, 104, 105],
    "product": ["Laptop", "Mobile", "Headphones", "Keyboard", "Monitor"],
    "category": ["Electronics", "Electronics", "Audio",
                 "Accessories", "Electronics"],
    "price": [60000, 30000, 5000, 3000, 15000]
})

orders = pd.DataFrame({
    "order_id": [1001, 1002, 1003, 1004, 1005],
    "customer_id": [1, 2, 3, 4, 5],
    "product_id": [101, 102, 103, 104, 105],
    "quantity": [1, 2, 3, 2, 1],
    "order_date": [
        "2026-08-01", "2026-08-02", "2026-08-03",
        "2026-08-04", "2026-08-05"
    ]
})

payments = pd.DataFrame({
    "payment_id": [501, 502, 503, 504, 505],
    "order_id": [1001, 1002, 1003, 1004, 1005],
    "amount": [60000, 60000, 15000, 6000, 15000],
    "status": ["Paid", "Paid", "Paid", "Pending", "Paid"]
})

# Save CSV files
customers.to_csv("customers.csv", index=False)
products.to_csv("products.csv", index=False)
orders.to_csv("orders.csv", index=False)
payments.to_csv("payments.csv", index=False)

# 2. Extract data
customers = pd.read_csv("customers.csv")
products = pd.read_csv("products.csv")
orders = pd.read_csv("orders.csv")
payments = pd.read_csv("payments.csv")
print("\nCUSTOMERS")
print(customers)
print("\nPRODUCTS")
print(products)
print("\nORDERS")
print(orders)
# 3. Data cleaning
customers = customers.drop_duplicates().dropna()
products = products.drop_duplicates().dropna()
orders = orders.drop_duplicates().dropna()
payments = payments.drop_duplicates().dropna()

orders["order_date"] = pd.to_datetime(orders["order_date"])
orders = orders[orders["quantity"] > 0]
print("\nDATA CLEANING COMPLETED")

# 4. Transform data
report = orders.merge(products, on="product_id")
report = report.merge(customers, on="customer_id")

report["total_amount"] = report["quantity"] * report["price"]

print("\nTRANSFORMED DATA")
print(report)

# 5. Load into SQL database
conn = sqlite3.connect("ecommerce.db")

customers.to_sql("customers", conn, if_exists="replace", index=False)
products.to_sql("products", conn, if_exists="replace", index=False)
orders.to_sql("orders", conn, if_exists="replace", index=False)
payments.to_sql("payments", conn, if_exists="replace", index=False)

print("\nDATA LOADED INTO SQL DATABASE")

# 6. Historical data

historical = orders[
    orders["order_date"] < "2026-08-04"
]

print("\nHISTORICAL ORDERS")
print(historical)

# 7. Newly arriving data

new_order = pd.DataFrame({
    "order_id": [1006],
    "customer_id": [2],
    "product_id": [101],
    "quantity": [1],
    "order_date": ["2026-08-06"]
})

existing = pd.read_sql(
    "SELECT order_id FROM orders", conn
)["order_id"]

new_order = new_order[
    ~new_order["order_id"].isin(existing)
]

new_order.to_sql(
    "orders",
    conn,
    if_exists="append",
    index=False
)

print("\nNEWLY ARRIVING DATA")
print(new_order)

# 8. Generate report

sales_report = pd.read_sql("""
SELECT
    p.category,
    SUM(o.quantity) AS total_quantity,
    SUM(o.quantity * p.price) AS total_sales
FROM orders o
JOIN products p
ON o.product_id = p.product_id
GROUP BY p.category
""", conn)

print("\nE-COMMERCE SALES REPORT")
print(sales_report)

# 9. Save report

sales_report.to_csv(
    "ecommerce_report.csv",
    index=False
)
print("\nReport saved as ecommerce_report.csv")
conn.close()
print("\nETL PIPELINE COMPLETED SUCCESSFULLY!")



CUSTOMERS
   customer_id   name    city
0            1   Amit  Mumbai
1            2  Priya    Pune
2            3  Rahul   Delhi
3            4   Neha  Nashik
4            5  Rohan  Mumbai

PRODUCTS
   product_id     product     category  price
0         101      Laptop  Electronics  60000
1         102      Mobile  Electronics  30000
2         103  Headphones        Audio   5000
3         104    Keyboard  Accessories   3000
4         105     Monitor  Electronics  15000

ORDERS
   order_id  customer_id  product_id  quantity  order_date
0      1001            1         101         1  2026-08-01
1      1002            2         102         2  2026-08-02
2      1003            3         103         3  2026-08-03
3      1004            4         104         2  2026-08-04
4      1005            5         105         1  2026-08-05

DATA CLEANING COMPLETED

TRANSFORMED DATA
   order_id  customer_id  product_id  quantity order_date     product  \
0      1001            1         101         